# In-class exercise 6a — SOLUTION: Corpus Pre-processing and Dictionary Method

## The Moral Foundation Dictionary 

In this exercise, we will practice pre-processing natural language and use a very simple dictionary method to explore to what extend the American democrats and republican presidential candidates appeal to different **moral values**. 

### The moral foundations

A group of psychologists - notably Jonathan Halddt and Jesse Graham - noticed that human beings across different culture background often value similar virtues - such as careness, fairness, loyalty and authority - and theorized that these common themes might have been essential for humanity's survival. They developed the **moral foundation theory** (https://moralfoundations.org) which established the five pillers of morality that are shared among cultures: 

1. **Careness**: sympathy, mutual aid and nurturance. It helps human to care about the small and the weak and survives as a group.
2. **Fairness**: reciprocity and proportionality. It encourages production and enables justice and right. 
3. **Loyalty/In-group**: feeling like belonging to a group or a tribe, a difference between us and them. It encourages contributing to the collective and formation of coalitions.
4. **Authority**: observance of a storng leader, doing as orders command. It makes cooperation more efficient. 
5. **Purity/Sanctity**: disgust of filth and contamination - including physical ones, such as blood or dead body,  and religious ones ,such as incest. It improves the sanitary condition in human habitat and promotes self-control and spirituality.

We call the words that describe things promoted by the five ideas "virtues" and the opposites - negligence, freeloading, betrayal, disobedience and sacrilege - "vices". 

Researchers in social sciences often notice that different political factions appeal to different moral foundations. In the context of American politics for example, the Democrats emphasizes more careness and fairness, while the Republicans emphasizes loyalty and authority. 

We will work on a small exercise today, inspired by but not identical with Hackenburg, Brady and Tsakiris (2023, https://doi.org/10.1093/pnasnexus/pgad189). The data are downloaded from OSF (10.17605/OSF.IO/FZ6KP)

In [1]:
# Import everything we need today 

# The basics
from pathlib import Path
import pandas as pd

# NLP tool kit
import nltk
from nltk.stem import SnowballStemmer
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab')      
nltk.download('wordnet')    
nltk.download('omw-1.4') 
nltk.download('averaged_perceptron_tagger_eng')
from nltk.corpus import wordnet
from string import punctuation
from nltk.corpus import stopwords
from nltk import word_tokenize, pos_tag

# Regular expressions
import re

# Language detection 
import langdetect

import fastparquet



[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:997)>
[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:997)>
[nltk_data] Error loading omw-1.4: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:997)>


[nltk_data] Error loading averaged_perceptron_tagger_eng: <urlopen
[nltk_data]     error [SSL: CERTIFICATE_VERIFY_FAILED] certificate
[nltk_data]     verify failed: unable to get local issuer certificate
[nltk_data]     (_ssl.c:997)>


## Exercise 1: Import The Moral Dictionary 

Some words describing each virtue and vice is included data/moral_foundation_dictionary. 

1. Open the `mfd2.0.dic` in this folder with a text reader (VSC works well) and observe the structure. 
2. Read this document into python, remove the headers (line 1 to 12) so that it can be converted to a data frame. 
2. Convert `mfd2.0.dic` in python to a pandas data frame, where one column is the word, another column is the code representing its categories (e.g. "1" for care.virtue, 2 for care.vice). Name the two columns `word` and `category`. 
3. Replace the category column with its label from the MFD, so that instead of 1, the column now has a value of "care.virtue" (see exercise in the pandas session for reference)
4. Not necessary but good data practice: put the columns to the good category (str for word and categorical for category.)


Hints: 
- You might meet all kinds of text materials in different formats in NLP exercises. Most of them can be opened by VSC as text, and read as text. `read().splitlines()` and `read().split()` are your good friends. 
- Here we are working in English so special characters are not a problem. If you are working with French, its a good practice to always read with the accurated encoding (e.f. "utf-8") to make sure that accented characters are visible. 
- "\t" means space and "\n" means line skip. 


In [2]:
# imports - here are some useful paths
this_dir = Path(".")
moral_foundation = this_dir / "data" / "moral_foundation_dictionary"
word_list = moral_foundation / "mfd2.0.dic"
word_list.exists()


True

In [3]:
# Read the file line by line (splitlines handles any kind of line break)
with open(word_list, encoding='utf-8') as file:
    lines = file.read().splitlines()

# Lines 1 to 12 are the header: '%', the 10 "code<TAB>label" lines, '%'
header = lines[:12]
print(header)

# Keep the code -> label mapping from the header, e.g. {'1': 'care.virtue', ...}
categories = dict(line.split('\t') for line in header[1:11])

# The rest are "word<TAB>code" lines (some entries are expressions, e.g. "civil rights")
body = [line for line in lines[12:] if line.strip() != '']
mfd = pd.DataFrame([line.rsplit('\t', 1) for line in body], columns=['word', 'category'])
mfd['word'] = mfd['word'].str.strip().str.lower()
mfd['category'] = mfd['category'].str.strip()

# Replace the code by its label
mfd['category'] = mfd['category'].map(categories)

# Good data practice: correct types
mfd['word'] = mfd['word'].astype(str)
mfd['category'] = pd.Categorical(mfd['category'], categories=list(categories.values()))

print(mfd.shape)
print(mfd['category'].value_counts(sort=False))
mfd.head()

['%', '1\tcare.virtue', '2\tcare.vice', '3\tfairness.virtue', '4\tfairness.vice', '5\tloyalty.virtue', '6\tloyalty.vice', '7\tauthority.virtue', '8\tauthority.vice', '9\tsanctity.virtue', '10\tsanctity.vice', '%']
(2104, 2)
category
care.virtue         182
care.vice           288
fairness.virtue     115
fairness.vice       236
loyalty.virtue      143
loyalty.vice         49
authority.virtue    301
authority.vice      130
sanctity.virtue     272
sanctity.vice       388
Name: count, dtype: int64


,word,category
0,compassion,care.virtue
1,empathy,care.virtue
2,kindness,care.virtue
3,caring,care.virtue
4,generosity,care.virtue


## Exercise 2: Stemming/Lemmatizing the words

### Try stemming and lemmatizing
We notice that each word comes in different grammar forms (such as 'caring', 'cared' and 'cares'). We would like to shrink the size of our dictionary, so that any program would run faster. 

For that we have two choices: stemming and lemmatizing. 

1. Produce a vector of stemmed moral keywords - be careful not to overwrite the column in the data frame yet. We will use the `SnowballStemmer` from `nltk` as in the lecture. 
2. Produce a vector of lemmatized moral keywords - be careful not to overwrite the column in the data frame yet. We will use the `WordNetLemmatizer` from `nltk` as in the lecture. 

In [4]:
stemmer = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()

# Some entries are expressions ("civil rights"), so we stem/lemmatize word by word
def stem_entry(entry):
    return " ".join(stemmer.stem(w) for w in entry.split())

def lemmatize_entry(entry):
    return " ".join(lemmatizer.lemmatize(w) for w in entry.split())

# Vectors only - the data frame is not modified yet
stemmed_words = mfd['word'].apply(stem_entry)
lemmatized_words = mfd['word'].apply(lemmatize_entry)

comparison = pd.DataFrame({'original': mfd['word'],
                           'stemmed': stemmed_words,
                           'lemmatized': lemmatized_words})

print("unique original  :", mfd['word'].nunique())
print("unique stemmed   :", stemmed_words.nunique())
print("unique lemmatized:", lemmatized_words.nunique())
comparison.sample(15, random_state=1)

unique original  : 2041
unique stemmed   : 874
unique lemmatized: 1667


,original,stemmed,lemmatized
1283,underling,underl,underling
1409,treacherous,treacher,treacherous
368,ached,ach,ached
573,square dealers,squar dealer,square dealer
1237,ranking,rank,ranking
613,betrayal,betray,betrayal
774,stolen,stolen,stolen
1000,backstabs,backstab,backstabs
995,unfaithfulness,unfaith,unfaithfulness
824,loyal,loyal,loyal


### (Might be a problem): why is my lemmatizer not working? 
Now, you might notice that the lemmatizer is "barely altering" your words! For example, "caring" was not changed to "care". Why is this? 

Try to find out the solution on your favoriate search engine. Hint: it has something to do with how lemmatizer understand the word - the part-of-speech that it assigned to the word. 


**Answer.** By default `WordNetLemmatizer.lemmatize(word)` assumes that every word is a **noun** (`pos='n'`). "caring", "cared", "hurting"... are verb forms, and as nouns they have no other lemma, so they are returned unchanged. The fix is to give the lemmatizer the right part-of-speech: we tag the words with `nltk.pos_tag` (Penn Treebank tags such as `VBG`, `NN`, `JJ`) and translate the tag into the WordNet categories (`wordnet.VERB`, `wordnet.NOUN`, `wordnet.ADJ`, `wordnet.ADV`).

Caveat: here the words are tagged out of context (one dictionary entry at a time), so the tagger sometimes guesses wrong; in the tweets (Exercise 4) we will tag whole sentences, which is more accurate.

In [5]:
# Translate a Penn Treebank tag (from pos_tag) into a WordNet part-of-speech
def penn_to_wordnet(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    if tag.startswith('V'):
        return wordnet.VERB
    if tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

def lemmatize_entry_pos(entry):
    tagged = pos_tag(entry.split())
    return " ".join(lemmatizer.lemmatize(w, penn_to_wordnet(tag)) for w, tag in tagged)

# Example: without / with the part-of-speech
for w in ['caring', 'cared', 'cares', 'hurting', 'betrayed']:
    print(f"{w:10} -> default: {lemmatizer.lemmatize(w):10} with POS: {lemmatize_entry_pos(w)}")

lemmatized_pos_words = mfd['word'].apply(lemmatize_entry_pos)
comparison['lemmatized_pos'] = lemmatized_pos_words
print("\nunique lemmatized with POS:", lemmatized_pos_words.nunique())
comparison.sample(15, random_state=1)

caring     -> default: caring     with POS: care
cared      -> default: cared      with POS: care
cares      -> default: care       with POS: care
hurting    -> default: hurting    with POS: hurt
betrayed   -> default: betrayed   with POS: betrayed

unique lemmatized with POS: 1306


,original,stemmed,lemmatized,lemmatized_pos
1283,underling,underl,underling,underling
1409,treacherous,treacher,treacherous,treacherous
368,ached,ach,ached,ache
573,square dealers,squar dealer,square dealer,square dealer
1237,ranking,rank,ranking,rank
613,betrayal,betray,betrayal,betrayal
774,stolen,stolen,stolen,steal
1000,backstabs,backstab,backstabs,backstabs
995,unfaithfulness,unfaith,unfaithfulness,unfaithfulness
824,loyal,loyal,loyal,loyal


### Pick an option 

Given the results and our goal (count occurences of these words in natural speech = Twitter), do you prefer stemming or lemmatizing? 

There is no correct answer, just explain your logic. 

**Answer.** For counting moral words in tweets, both are defensible:

- **Stemming** shrinks the dictionary the most and is very robust: "caring", "cared", "cares" all become "care", whatever the context. But it is crude: stems are often not real words ("compassion" → "compass", "generosity" → "generos"), and unrelated words can collapse onto the same stem (e.g. "compassion" and "compass", "organ" and "organization"), which would create **false positives** when we count in tweets.
- **Lemmatizing** (with the POS tag) returns real dictionary words, so the dictionary stays readable and interpretable, and it does not merge words with different meanings. It reduces the dictionary a bit less and depends on the POS tagger, which can be wrong on short, noisy texts like tweets.

Since the MFD is a carefully chosen list of words whose *meaning* matters, I prefer **lemmatizing**: we lose a bit of recall but we avoid counting words that are not really moral words. The key point is to apply **exactly the same** transformation to the dictionary and to the tweets.

### We adopt the lemmatizer 

For the rest of the exercise we will keep the lemmatizer. Please replace the `word` column of your data frame `mfd` with the lemmatized version, and then drop the duplicates. (`drop_duplicates()` is helpful here )

In [6]:
mfd['word'] = lemmatized_pos_words
mfd = mfd.drop_duplicates().reset_index(drop=True)

print(mfd.shape)
mfd.head()

(1348, 2)


,word,category
0,compassion,care.virtue
1,empathy,care.virtue
2,kindness,care.virtue
3,care,care.virtue
4,generosity,care.virtue


## Exercise 3: Importing Tweets from presidential candidates 

We would like to pre-process our presendital candidate tweets right now! We are looking at 4 sets of tweets: Biden, Clinton, Trump 2016 and Trump 2020. Similarly, we would like to read it into a pandas DF with the name of the candidate and the tweet.

We would ideally like to have a pandas dataframe with three columns: `candidate`, `year` and `original_text`. We will call the dataframe `all_tweets`.

Here is how I proceeded to read the files, please complete the code so that you manage to read the dataframe (hint: look at what variables have not been defined) but feel free to do otherwise! 

1. create an empty data frame with the desired columns. 
2. iterrate files in the directorate 
3. for each directorate, extracte "2016" or "2020" in the file name for year and the name of the candidate. Put them in variables to be inserted later. 
4. read the file by line and convert it to a dataframe. 
5. create the year and candidate columns using stored variables. 
6. append (`pd.concat`) the dataframe read from one file to the main file `all_tweets`. 


P.S. Some of Trump's tweet looks "unfinished" in the original file. This is the way I found it at download - it is ok for the purpose of the exercise. 

In [7]:
tweet_data = this_dir / "data" / "politician_tweets"

# 1. empty data frame with the desired columns
all_tweets = pd.DataFrame(columns=['candidate', 'year', 'original_text'])

# 2. iterate over the files (sorted, so that the order is the same on every computer)
for dir in sorted(tweet_data.iterdir()):

    if dir.suffix != '.txt':   # skip hidden files such as .DS_Store
        continue

    # 3. year and candidate from the file name
    if dir.name.find("2016") > 0 :
        year = 2016
    elif dir.name.find("2020") > 0 :
        year = 2020

    for candidate in ['biden', 'hillary', 'trump']:
        if dir.name.find(candidate) > 0 :
            person = candidate

    # 4. read the file line by line
    with open(dir, encoding='utf-8') as file:
        tweets = file.read().splitlines()

    # 5. create the columns
    tweets = pd.DataFrame(tweets)
    tweets.columns = ['original_text']
    tweets['candidate'] = person
    tweets['year'] = year

    # 6. append
    all_tweets = pd.concat([all_tweets, tweets])

# Put them into the correct category
all_tweets['candidate'] = all_tweets['candidate'].astype('category')
all_tweets['year'] = all_tweets['year'].astype(int)
all_tweets['original_text'] = all_tweets['original_text'].astype(str)

# don't foget to reset index!
all_tweets = all_tweets.reset_index(drop=True)

print(all_tweets.groupby(['candidate', 'year'], observed=True).size())
all_tweets

candidate  year
biden      2020    2286
hillary    2016    4112
trump      2016    2233
           2020    8662
dtype: int64


,candidate,year,original_text
0,trump,2016,Just landed in Iowa - speaking soon!
1,trump,2016,It is time to
2,trump,2016,Thank you!
3,trump,2016,Will be on at 7:02 A.M. Enjoy.
4,trump,2016,
...,...,...,...
17288,hillary,2016,"""Hillary Clinton is the better candidate to ta..."
17289,hillary,2016,72 years after let's not just eulogize the br...
17290,hillary,2016,Donald Trump should come out of the towers he ...
17291,hillary,2016,bigoted comments about a Latino judge are so ...


(we can insert an exercise on sentiment analysis or part-of-word here, we would also add purging spanish) 

## Exercise 4: Pre-processing the tweets

We would like to do the following things: 
- put everything to lower case
- remove all non-alphabetic things
- remove punctuation 
- remove stopwords
- lemmatizing 

1. Please create a function called `noralize_text` that would take a document (here, a tweet) and return a list of lemmatized words. Feel free to copy-paste from the lecture and see what happens. 
2. Then, `apply` it to the column `original_text` to generate a column called `processed_text`.

In [8]:
nltk.download('stopwords')   # the stopword list is not downloaded in the intro cell

translator = str.maketrans('', '', punctuation)
stoplist = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def normalize_text(doc):
    "Input doc and return clean list of lemmatized tokens"
    doc = doc.replace('\r', ' ').replace('\n', ' ')
    lower = doc.lower()                                   # all lower case
    nopunc = lower.translate(translator)                  # remove punctuation
    words = nopunc.split()                                # split into tokens
    nostop = [w for w in words if w not in stoplist]      # remove stopwords
    alpha = [w for w in nostop if w.isalpha()]            # remove non-alphabetic tokens (numbers, '7:02'...)
    tagged = pos_tag(alpha)                               # POS of each word, in the context of the tweet
    return [lemmatizer.lemmatize(w, penn_to_wordnet(tag)) for w, tag in tagged]

print(normalize_text("Just landed in Iowa - speaking soon!"))

all_tweets['processed_text'] = all_tweets['original_text'].apply(normalize_text)

# Empty lines of the original files give an empty list: they are not tweets, drop them
all_tweets = all_tweets[all_tweets['processed_text'].str.len() > 0].reset_index(drop=True)
all_tweets[['original_text', 'processed_text']].head(10)

['land', 'iowa', 'speaking', 'soon']


[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:997)>


,original_text,processed_text
0,Just landed in Iowa - speaking soon!,"[land, iowa, speaking, soon]"
1,It is time to,[time]
2,Thank you!,[thank]
3,Will be on at 7:02 A.M. Enjoy.,[enjoy]
4,Mexico will pay for the wall!,"[mexico, pay, wall]"
5,Watching the,[watch]
6,I will be on at 8:00 P.M. Enjoy!,"[pm, enjoy]"
7,Trump promises special session to repeal Obama...,"[trump, promise, special, session, repeal, oba..."
8,Yet more evidence of a media-rigged election:,"[yet, evidence, mediarigged, election]"
9,"is doing a great job - so far, no contest!","[great, job, far, contest]"


### Bonus: Can I adapt the off-the-shelf pre-processing function to better fit our use case? 

If you had copy-pasted the function made by Malka in the lecture, you might want to circle a bit in the dataset and see whether it has problems - whether it failed to process certain words that should be processed. 

Here are some of the questions that I noticed: 
- emojis are not purged
- it failed to separate contractions (we've, haven't)...
- it failed to separate words with punctuation in the middle (e-mail, anti-vax, a point after a word with no space, etc)
- it failed to remove extremely common words (we've, us, etc)
- it failed to discern other languages (spanish, notably)

Can you improve the function so that it takes care of these? 

Hints: 
- Emojis are ususally considered unicodes by python. They can be purged with regular expressions : try `[\U00010000-\U0010FFFF]`
- For contractions, try to spilt further by the apostrophe '
- For Spanish, try a language recogniser. (check out what is imported in the intro! )
- also purging some other extremely common stopwords might be a good idea. here is my list : 

`extra_stopwords = ['us', 'we', 've', 'i', 'you', 'he', 'she', 'it', 'they', 'me', 'him', 'her', 'them', 'my', 'your', 'his', 'its', 'our', 'their', 'a', 'an', 'the', 'and', 'or', 'but', 'if', 'in', 'on', 'at', 'to', 'for', 'with', 'of', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'get', 'let']`


You are welcomed (and encouarged) to fix problems that I didn't find!

In [9]:
from langdetect import DetectorFactory, detect_langs
from langdetect.lang_detect_exception import LangDetectException
DetectorFactory.seed = 0   # langdetect is random, fix the seed to get reproducible results

extra_stopwords = ['us', 'we', 've', 'i', 'you', 'he', 'she', 'it', 'they', 'me', 'him', 'her', 'them', 'my', 'your', 'his', 'its', 'our', 'their', 'a', 'an', 'the', 'and', 'or', 'but', 'if', 'in', 'on', 'at', 'to', 'for', 'with', 'of', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'get', 'let']
stoplist_extended = stoplist | set(extra_stopwords)

url_pattern = re.compile(r'http\S+|www\.\S+')
emoji_pattern = re.compile('[\U00010000-\U0010FFFF☀-➿️]')

def is_spanish(doc):
    "True if langdetect is confident that the tweet is in Spanish"
    # langdetect is unreliable on very short texts, so we only test tweets with a few words
    if len(doc.split()) < 4:
        return False
    try:
        best = detect_langs(doc)[0]
    except LangDetectException:
        return False
    return best.lang == 'es' and best.prob > 0.9

def normalize_text_v2(doc):
    "Improved version: emojis, contractions, glued words, extra stopwords and Spanish"
    if is_spanish(doc):
        return []
    doc = doc.replace('\r', ' ').replace('\n', ' ')
    doc = url_pattern.sub(' ', doc)                       # remove links
    doc = emoji_pattern.sub(' ', doc)                     # remove emojis
    lower = doc.lower()
    # Replace everything that is not a letter by a space: this splits contractions
    # (we've -> we ve, haven’t -> haven t), words glued by punctuation (anti-vax -> anti vax,
    # "end.Next" -> end next) and removes numbers and non-latin characters
    letters_only = re.sub(r'[^a-z]+', ' ', lower)
    words = [w for w in letters_only.split() if len(w) > 1 and w not in stoplist_extended]
    tagged = pos_tag(words)
    lemmas = [lemmatizer.lemmatize(w, penn_to_wordnet(tag)) for w, tag in tagged]
    # lemmatizing can create new stopwords ("got" -> "get"), so we filter again
    return [w for w in lemmas if w not in stoplist_extended]

examples = ["We’ve got to protect our e-mail from anti-vax people 🙏🇺🇸",
            "We haven't finished.Next step: vote!",
            "We just won Puerto Rico! ¡Gracias a la Isla del Encanto por su apoyo!",
            "Hoy votamos por un futuro mejor para nuestras familias"]
for ex in examples:
    print(f"{ex}\n   v1: {normalize_text(ex)}\n   v2: {normalize_text_v2(ex)}")

We’ve got to protect our e-mail from anti-vax people 🙏🇺🇸
   v1: ['get', 'protect', 'email', 'antivax', 'people']
   v2: ['protect', 'mail', 'anti', 'vax', 'people']
We haven't finished.Next step: vote!
   v1: ['havent', 'finishednext', 'step', 'vote']
   v2: ['finish', 'next', 'step', 'vote']
We just won Puerto Rico! ¡Gracias a la Isla del Encanto por su apoyo!
   v1: ['puerto', 'rico', 'la', 'isla', 'del', 'encanto', 'por', 'su', 'apoyo']
   v2: []
Hoy votamos por un futuro mejor para nuestras familias
   v1: ['hoy', 'votamos', 'por', 'un', 'futuro', 'mejor', 'para', 'nuestras', 'familias']
   v2: []


In [10]:
# We keep the improved version for the rest of the exercise
all_tweets['processed_text'] = all_tweets['original_text'].apply(normalize_text_v2)

# Tweets in Spanish (and tweets with only stopwords) give an empty list: they are not
# English tweets, so we drop them - otherwise they would bias the "per tweet" averages downward
print("empty processed tweets:", (all_tweets['processed_text'].str.len() == 0).sum())
all_tweets = all_tweets[all_tweets['processed_text'].str.len() > 0].reset_index(drop=True)

print(all_tweets.groupby(['candidate', 'year'], observed=True).size())
all_tweets[['original_text', 'processed_text']].sample(10, random_state=3)

empty processed tweets: 97
candidate  year
biden      2020    2259
hillary    2016    4025
trump      2016    2179
           2020    7770
dtype: int64


,original_text,processed_text
5314,Ky. Attorney General Cameron: President Trump...,"[ky, attorney, general, cameron, president, tr..."
12700,A (non-exhaustive) list of all the terrible th...,"[non, exhaustive, list, terrible, thing, donal..."
13567,Trump's plan:Cut taxes for billionaires ; mill...,"[trump, plan, cut, tax, billionaire, millionai..."
15770,"""Let me tell you, my faith in Hillary Clinton ...","[tell, faith, hillary, clinton, always, reward]"
11504,This is a job Donald Trump is entirely unfit f...,"[job, donald, trump, entirely, unfit]"
7651,Thank you. United We Stand!,"[thank, united, stand]"
1386,Great job,"[great, job]"
4619,The administration’s coronavirus response has...,"[administration, coronavirus, response, delive..."
8212,Final RCP Polling Averages Had Hillary Clinton...,"[final, rcp, polling, average, hillary, clinto..."
6249,NEW: FBI Received Evidence Of Russian Disinfo...,"[new, fbi, receive, evidence, russian, disinfo..."


## Exercise 5: count dictionary 

Now, we would like to count the frequencies of different virtues keyword and store them as additional variables in the data frame `all_tweets`. 

1. Create a function called `count_virtue` that would count that takes a category of virtue or vice (e.g. `"care.virtue"`) and return a count of the # of occurences of any virtue key word in the tweet. 
2. Apply it to the tweets for each virtue and vice, and add them to the variable as additional columns (e.g. `count_care_virtue`)

Hints: 

- Try to make your function self-contained; the data frame `all_tweets` and the dictionary `mfd` should not be pulled from the global environment but rather given as inputs. 
- If your function counts at the level of a single string, the function `apply` is your friend. 


In [11]:
def count_virtue(tokens, category, mfd):
    """Number of keywords of `category` (e.g. "care.virtue") in a processed tweet (list of tokens).

    Single-word entries are matched token by token, multi-word entries (e.g. "civil right")
    are matched as a sequence of consecutive tokens.
    """
    keywords = mfd.loc[mfd['category'] == category, 'word']
    single_words = set(k for k in keywords if ' ' not in k)
    expressions = [k for k in keywords if ' ' in k]

    count = sum(token in single_words for token in tokens)
    text = " " + " ".join(tokens) + " "
    count += sum(text.count(" " + expr + " ") for expr in expressions)
    return count

print(count_virtue(['protect', 'family', 'care', 'love'], 'care.virtue', mfd))

for category in mfd['category'].cat.categories:
    column = 'count_' + category.replace('.', '_')
    all_tweets[column] = all_tweets['processed_text'].apply(count_virtue, args=(category, mfd))

count_columns = [c for c in all_tweets.columns if c.startswith('count_')]
all_tweets[['original_text'] + count_columns].head()

3


,original_text,count_care_virtue,count_care_vice,count_fairness_virtue,count_fairness_vice,count_loyalty_virtue,count_loyalty_vice,count_authority_virtue,count_authority_vice,count_sanctity_virtue,count_sanctity_vice
0,Just landed in Iowa - speaking soon!,0,0,0,0,0,0,0,0,0,0
1,It is time to,0,0,0,0,0,0,0,0,0,0
2,Thank you!,0,0,0,0,0,0,0,0,0,0
3,Will be on at 7:02 A.M. Enjoy.,0,0,0,0,0,0,0,0,0,0
4,Mexico will pay for the wall!,0,0,0,0,0,0,0,0,0,0


## Exercise 6: Who says it more? 

Finally, we would like to test the hypothesis that democrats emphasizes more the care and fairness virtures, while Republicans (Trump here) emphasizes more authority and in-groupness. Of course, we must normalize by the total sum of tweets - trump is just tweeting more than other people! 

Use pandas' `.groupby` and `.agg` functions to calculate the mean number of words about a virtue or vice per tweet emitted by each candidate/year. What do you think? 


In [12]:
# Mean number of moral words per tweet, by candidate/year
mean_per_tweet = (all_tweets
                  .groupby(['candidate', 'year'], observed=True)[count_columns]
                  .agg('mean'))
mean_per_tweet.round(3)

count_care_virtue  count_care_vice  count_fairness_virtue  \
candidate year                                                              
biden     2020              0.375            0.175                  0.161   
hillary   2016              0.152            0.089                  0.110   
trump     2016              0.084            0.072                  0.041   
          2020              0.161            0.095                  0.075   

                count_fairness_vice  count_loyalty_virtue  count_loyalty_vice  \
candidate year                                                                  
biden     2020                0.041                 0.503               0.003   
hillary   2016                0.037                 0.201               0.001   
trump     2016                0.177                 0.177               0.005   
          2020                0.055                 0.179               0.008   

                count_authority_virtue  count_authority_vice  \
candidate year                                                 
biden     2020                   0.459                 0.023   
hillary   2016                   0.196                 0.011   
trump     2016                   0.145                 0.023   
          2020                   0.292                 0.061   

                count_sanctity_virtue  count_sanctity_vice  
candidate year                                              
biden     2020                  0.098                0.110  
hillary   2016                  0.024                0.014  
trump     2016                  0.021                0.027  
          2020                  0.045                0.073

In [13]:
# Same thing for the hypothesis: Democrats -> care & fairness, Republicans -> loyalty & authority
# (expressed per 100 tweets to make it easier to read)
hypothesis = pd.DataFrame({
    'care + fairness (virtue)': mean_per_tweet['count_care_virtue'] + mean_per_tweet['count_fairness_virtue'],
    'loyalty + authority (virtue)': mean_per_tweet['count_loyalty_virtue'] + mean_per_tweet['count_authority_virtue'],
}) * 100
hypothesis.round(1)

care + fairness (virtue)  loyalty + authority (virtue)
candidate year                                                        
biden     2020                      53.5                          96.3
hillary   2016                      26.2                          39.7
trump     2016                      12.5                          32.2
          2020                      23.6                          47.1

In [14]:
# Robustness: tweets do not all have the same length, so we can also normalize by the number of words
# (moral words per 100 processed words)
n_words = all_tweets.groupby(['candidate', 'year'], observed=True)['processed_text'].agg(lambda s: s.str.len().sum())
per_100_words = (all_tweets.groupby(['candidate', 'year'], observed=True)[count_columns].agg('sum')
                 .div(n_words, axis=0) * 100)
per_100_words.round(2)

count_care_virtue  count_care_vice  count_fairness_virtue  \
candidate year                                                              
biden     2020               2.61             1.22                   1.12   
hillary   2016               1.76             1.03                   1.27   
trump     2016               0.98             0.84                   0.48   
          2020               1.43             0.85                   0.66   

                count_fairness_vice  count_loyalty_virtue  count_loyalty_vice  \
candidate year                                                                  
biden     2020                 0.28                  3.50                0.02   
hillary   2016                 0.43                  2.33                0.02   
trump     2016                 2.06                  2.06                0.05   
          2020                 0.49                  1.59                0.08   

                count_authority_virtue  count_authority_vice  \
candidate year                                                 
biden     2020                    3.20                  0.16   
hillary   2016                    2.26                  0.13   
trump     2016                    1.69                  0.27   
          2020                    2.59                  0.54   

                count_sanctity_virtue  count_sanctity_vice  
candidate year                                              
biden     2020                   0.68                 0.77  
hillary   2016                   0.28                 0.16  
trump     2016                   0.24                 0.31  
          2020                   0.40                 0.65

In [15]:
# Which words drive the counts?
from collections import Counter

def top_matches(tokens_series, category, mfd, n=8):
    keywords = set(mfd.loc[mfd['category'] == category, 'word'])
    return Counter(t for tokens in tokens_series for t in tokens if t in keywords).most_common(n)

for category in ['care.virtue', 'fairness.vice', 'loyalty.virtue', 'authority.virtue', 'authority.vice', 'sanctity.vice']:
    print(f"--- {category}")
    for (candidate, year), group in all_tweets.groupby(['candidate', 'year'], observed=True):
        print(f"{candidate} {year}: {top_matches(group['processed_text'], category, mfd)}")

--- care.virtue
biden 2020: [('help', 158), ('care', 141), ('health', 133), ('safe', 65), ('protect', 62), ('love', 38), ('share', 30), ('relief', 24)]
hillary 2016: [('help', 140), ('child', 69), ('love', 67), ('care', 62), ('health', 51), ('share', 39), ('protect', 32), ('mother', 22)]
trump 2016: [('safe', 38), ('love', 32), ('help', 24), ('protect', 17), ('care', 11), ('share', 10), ('child', 9), ('condolence', 7)]
trump 2020: [('help', 238), ('love', 164), ('protect', 137), ('care', 93), ('safe', 70), ('health', 53), ('hospital', 53), ('healthcare', 48)]
--- fairness.vice
biden 2020: [('lie', 26), ('racism', 22), ('disparity', 8), ('racist', 5), ('inequality', 4), ('oppression', 3), ('injustice', 3), ('betray', 2)]
hillary 2016: [('lie', 46), ('discrimination', 17), ('racist', 13), ('racism', 11), ('discriminate', 6), ('cheat', 6), ('injustice', 6), ('prejudice', 6)]
trump 2016: [('crook', 188), ('crooked', 45), ('dishonest', 40), ('lie', 38), ('fraud', 15), ('bias', 15), ('racist

In [16]:
# "president" is an authority.virtue keyword, but in a presidential campaign it is mostly a job title
# ("President Obama", "Vice President Biden"...). Robustness check: remove it from the dictionary
mfd_robust = mfd[~mfd['word'].isin(['president', 'presidential', 'vice president'])]
authority_robust = all_tweets['processed_text'].apply(count_virtue, args=('authority.virtue', mfd_robust))
pd.DataFrame({
    'authority.virtue per 100 tweets': mean_per_tweet['count_authority_virtue'] * 100,
    'without "president"': authority_robust.groupby([all_tweets['candidate'], all_tweets['year']], observed=True).mean() * 100,
}).round(1)

authority.virtue per 100 tweets  without "president"
candidate year                                                      
biden     2020                             45.9                 17.2
hillary   2016                             19.6                  9.5
trump     2016                             14.5                  8.9
          2020                             29.2                 16.6

**What do we find?**

*Care and fairness (Democrats' side of the hypothesis).* This part holds up well. Biden (0.375 care.virtue words per tweet) and Clinton (0.152) use more care words than Trump (0.084 in 2016, 0.161 in 2020). The same goes for fairness.virtue (Biden 0.161, Clinton 0.110, Trump 0.041 / 0.075). The ranking stays the same when we normalize by the number of words instead of the number of tweets (care.virtue per 100 words: Biden 2.6, Clinton 1.8, Trump 1.0 / 1.4). The top words make sense too: *help, care, health, protect, child, love*.

*Loyalty and authority (Republicans' side of the hypothesis).* This part is **not** supported. Biden has the highest loyalty.virtue and authority.virtue counts, Clinton is above Trump on loyalty.virtue and sits between Trump 2016 and Trump 2020 on authority.virtue. Two reasons:
- The dictionary picks up campaign vocabulary that is not really moral language. In authority.virtue, the most frequent match by far is **"president"** (Biden 621 times, Trump 2020 896 times): it is a job title ("President Obama", "Vice President Biden"), not deference to authority. Words like *governor*, *order* (as in "executive order") or *join* ("join us tonight") work the same way. Taking "president" out lowers authority.virtue for everyone, and Trump 2020 ends up close to Biden (16.6 vs. 17.2 per 100 tweets, see the robustness table).
- Biden's processed tweets are longer (about 14 words versus 9 to 11 for the others), so they contain more dictionary words of every kind. That is why we also normalized per word.

*Vices are more informative for Trump.* Trump 2016 is far ahead on **fairness.vice** (0.177 per tweet versus about 0.04 for the Democrats; *crook(ed), dishonest, lie, fraud*: "Crooked Hillary"), and Trump 2020 leads on **authority.vice** (*anarchist, riot, rioter, illegal, chaos*: the "law and order" message of 2020; careful, the top match *order* is ambiguous and is also listed in authority.virtue). So Trump also appeals to fairness and authority, but in a negative way: he points at their violation. Note that sanctity.vice is high for both 2020 candidates because of COVID vocabulary (*pandemic, virus, epidemic, spread*), which again is topic rather than morality.

**Conclusion.** The dictionary method partly confirms the hypothesis (Democrats use more care and fairness language). For loyalty and authority, the results depend on a few ambiguous, context-dependent words, and on the topics of each campaign (2016 vs. 2020 and COVID). A simple word count cannot tell "President" from "respect for authority". To go further we would need to: check the most frequent matches and clean the dictionary for this setting, compare candidates within the same year, look at virtues and vices together, and test whether the differences are statistically significant (e.g. a regression of the counts on candidate dummies, controlling for tweet length).

## Save your pre-processed database as a .parquet 

Please save the columns `candidate`,  `year` , `original_text` and `processed_text` in a parquet document called "`cooked_data.parquet`" in the folder `data`. and the mfd dataframe as `cooked_mfd.parquet` in the same place. 

We will keep using this data for the next class! 

In [17]:
# Define a path 
path_cooked_data = this_dir / "data" / "cooked_data.parquet"

all_tweets[['candidate','year','original_text', 'processed_text']].to_parquet(path_cooked_data)


In [18]:
# Same for mfd
path_mfd_cooked = this_dir / "data" / "cooked_mfd.parquet"
mfd.to_parquet(path_mfd_cooked)

### If you still have time: Try other things we did in class!

If you still have some time at hand, please try the other things we mentioned today: 
- Sentiment analysis: calculate a positivity score for each tweet and aggregate. Which candidate is the most positive one ?
- Part-of-Speech; try Named entity recogination on some tweets and see what entity (e.g. "Donald Trump" "The Fed" "2016") are highlighted. 


In [19]:
# Sentiment analysis with VADER (as in the lecture), on the original text (VADER uses punctuation, caps, emojis)
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sent_analyzer = SentimentIntensityAnalyzer()

all_tweets['sentiment'] = all_tweets['original_text'].apply(lambda t: sent_analyzer.polarity_scores(t)['compound'])
all_tweets.groupby(['candidate', 'year'], observed=True)['sentiment'].agg(['mean', 'median']).round(3)

[nltk_data] Error loading vader_lexicon: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:997)>


mean  median
candidate year               
biden     2020  0.156   0.145
hillary   2016  0.121   0.000
trump     2016  0.136   0.273
          2020  0.145   0.095